# Taylor Swift Reddit Tracker

This notebook demonstrates tracking Taylor Swift content across multiple Reddit subreddits, linking posts to the Creator entity, and tagging posts by era.

## Features
- Fetch Reddit posts about Taylor Swift
- Link posts to Creator entity
- Tag posts by era (TLOAS, Folklore, Reputation, etc.)
- Track multiple subreddits as sources
- Query posts by subreddit and era


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread
from feed.storage.neo4j_connection import get_connection
from feed.utils.reddit_url_extractor import extract_post_id_from_url, parse_reddit_url
from feed.services.creator_service import CreatorService

# Connect to Neo4j
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}")


## Helper Function: Process Taylor Swift Post

This function handles all the Reddit-specific processing for Taylor Swift posts.


In [ ]:
def process_taylor_swift_post(post_url: str, era_tag: str = None, era_full_name: str = None):
    """
    Process a Reddit post about Taylor Swift, link it to the Creator entity, and tag with era.
    
    Args:
        post_url: Reddit post URL
        era_tag: Optional era tag (e.g., "TLOAS", "Folklore", "Reputation")
        era_full_name: Optional full era name
    """
    # Extract post ID
    post_id = extract_post_id_from_url(post_url)
    print(f"Processing post ID: {post_id}")
    
    # Check if post exists
    check_query = """
    MATCH (p:Post {id: $post_id})
    RETURN p.id as id, p.title as title, p.permalink as permalink, p.era as era
    """
    result = neo4j.execute_read(check_query, parameters={"post_id": post_id})
    
    if result:
        record = result[0]
        print(f"Post already exists in database:")
        print(f"  ID: {record['id']}")
        print(f"  Title: {record.get('title', 'N/A')}")
        print(f"  Era: {record.get('era', 'N/A')}")
        return
    
    print(f"Post not found in database. Fetching from Reddit...")
    
    # Fetch the post
    reddit = RedditAdapter()
    
    # Extract permalink from URL
    parsed = parse_reddit_url(post_url)
    if parsed and parsed.get('permalink'):
        permalink = parsed['permalink']
    else:
        from urllib.parse import urlparse
        parsed_url = urlparse(post_url)
        path_parts = parsed_url.path.strip('/').split('/')
        if len(path_parts) >= 4 and path_parts[0] == 'r' and path_parts[2] == 'comments':
            permalink = '/' + '/'.join(path_parts[:4]) + '/'
        else:
            permalink = None
    
    if not permalink:
        print("Could not extract permalink from URL")
        return
    
    print(f"Fetching thread: {permalink}")
    post, comments, raw_post_data = reddit.fetch_thread(permalink)
    
    if not post:
        print("Failed to fetch post from Reddit")
        return
    
    print(f"Fetched post:")
    print(f"  ID: {post.id}")
    print(f"  Title: {post.title}")
    print(f"  Subreddit: {post.subreddit}")
    print(f"  Comments: {len(comments)}")
    
    # Extract images
    images = []
    post_images = reddit.extract_all_images(post, raw_post_data)
    for img_url in post_images:
        images.append({
            "url": img_url,
            "source": "post",
            "post_id": post.id,
        })
    
    comment_images = reddit.extract_images_from_comments(comments)
    images.extend(comment_images)
    
    print(f"  Images found: {len(images)}")
    
    # Store in Neo4j
    print(f"Storing in Neo4j...")
    try:
        store_thread(neo4j, post, comments, images, raw_post_data)
        print(f"Successfully stored post {post.id} in database!")
        
        # Create/update Taylor Swift Creator entity
        creator_service = CreatorService()
        ts_creator = creator_service.get_or_create_creator(
            name="Taylor Swift",
            slug="taylor-swift"
        )
        print(f"Creator entity: {ts_creator.get('name', 'N/A')} (slug: {ts_creator.get('slug', 'N/A')})")
        
        # Link subreddit to Creator entity
        link_subreddit_query = """
        MATCH (c:Creator {slug: $creator_slug})
        MATCH (s:Subreddit {name: $subreddit})
        MERGE (c)-[r:HAS_SOURCE]->(s)
        ON CREATE SET 
            r.source_type = 'reddit',
            r.discovered_at = datetime(),
            r.created_at = datetime()
        ON MATCH SET
            r.updated_at = datetime()
        RETURN r
        """
        neo4j.execute_write(
            link_subreddit_query,
            parameters={
                "creator_slug": "taylor-swift",
                "subreddit": post.subreddit
            }
        )
        print(f"Linked subreddit r/{post.subreddit} to Taylor Swift Creator entity")
        
        # Tag post with era if provided
        if era_tag:
            tag_era_query = """
            MATCH (p:Post {id: $post_id})
            SET p.era = $era,
                p.era_full_name = $era_full_name,
                p.updated_at = datetime()
            RETURN p.id as id, p.era as era
            """
            era_result = neo4j.execute_write(
                tag_era_query,
                parameters={
                    "post_id": post.id,
                    "era": era_tag,
                    "era_full_name": era_full_name or era_tag
                }
            )
            if era_result:
                print(f"Tagged post with era: {era_tag}" + (f" ({era_full_name})" if era_full_name else ""))
        
        # Link post to Creator entity
        link_post_query = """
        MATCH (c:Creator {slug: $creator_slug})
        MATCH (p:Post {id: $post_id})
        MERGE (p)-[r:ABOUT]->(c)
        ON CREATE SET
            r.created_at = datetime()
        RETURN r
        """
        neo4j.execute_write(
            link_post_query,
            parameters={
                "creator_slug": "taylor-swift",
                "post_id": post.id
            }
        )
        print(f"Linked post to Taylor Swift Creator entity")
        
    except Exception as e:
        print(f"Error storing post: {e}")
        import traceback
        traceback.print_exc()


## Process Posts

Process Reddit posts about Taylor Swift from various subreddits.


In [ ]:
# Post 1: Golden Glam (TLOAS era)
post_url_1 = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1puk3m7/taylor_swift_in_golden_glam/"
process_taylor_swift_post(post_url_1, era_tag="TLOAS", era_full_name="The Tortured Poets Department")


In [ ]:
# Post 3: Award Show Performance Outfits (gallery post)
post_url_3 = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1oytjju/taylors_award_show_performance_outfits_through/"
process_taylor_swift_post(post_url_3, era_tag=None, era_full_name=None)


## Crawler URL Index

Check if we've seen a Reddit post URL before. This is useful for monitoring the crawler and preventing duplicate processing.


In [ ]:
def check_url_seen(post_url: str) -> dict:
    """
    Check if a Reddit post URL has been seen/processed before.
    
    Args:
        post_url: Reddit post URL
        
    Returns:
        Dictionary with 'seen' (bool), 'post_id', 'title', 'permalink', 'created', 'era' if found
    """
    post_id = extract_post_id_from_url(post_url)
    
    if not post_id:
        return {"seen": False, "error": "Could not extract post ID from URL"}
    
    # Check by post ID
    query = """
    MATCH (p:Post {id: $post_id})
    RETURN p.id as id, 
           p.title as title, 
           p.permalink as permalink,
           p.url as url,
           p.created_utc as created,
           p.era as era,
           p.era_full_name as era_full_name,
           p.subreddit as subreddit,
           p.score as score
    LIMIT 1
    """
    
    result = neo4j.execute_read(query, parameters={"post_id": post_id})
    
    if result:
        record = result[0]
        return {
            "seen": True,
            "post_id": record.get("id"),
            "title": record.get("title"),
            "permalink": record.get("permalink"),
            "url": record.get("url"),
            "created": record.get("created"),
            "era": record.get("era"),
            "era_full_name": record.get("era_full_name"),
            "subreddit": record.get("subreddit"),
            "score": record.get("score")
        }
    
    # Also check by URL (in case permalink is different)
    query_by_url = """
    MATCH (p:Post)
    WHERE p.url = $url OR p.permalink CONTAINS $post_id
    RETURN p.id as id, 
           p.title as title, 
           p.permalink as permalink,
           p.url as url,
           p.created_utc as created,
           p.era as era,
           p.subreddit as subreddit
    LIMIT 1
    """
    
    result_by_url = neo4j.execute_read(
        query_by_url, 
        parameters={"url": post_url, "post_id": post_id}
    )
    
    if result_by_url:
        record = result_by_url[0]
        return {
            "seen": True,
            "post_id": record.get("id"),
            "title": record.get("title"),
            "permalink": record.get("permalink"),
            "url": record.get("url"),
            "created": record.get("created"),
            "era": record.get("era"),
            "subreddit": record.get("subreddit")
        }
    
    return {"seen": False, "post_id": post_id}

# Test the function with the new post
test_url = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1oytjju/taylors_award_show_performance_outfits_through/"
result = check_url_seen(test_url)

if result.get("seen"):
    print(f"URL already seen:")
    print(f"  Post ID: {result.get('post_id')}")
    print(f"  Title: {result.get('title')}")
    print(f"  Subreddit: r/{result.get('subreddit')}")
    print(f"  Created: {result.get('created')}")
    print(f"  Era: {result.get('era', 'N/A')}")
    print(f"  Score: {result.get('score', 'N/A')}")
else:
    print(f"URL not seen before (Post ID: {result.get('post_id')})")
    print("You can process it using process_taylor_swift_post()")


## Batch Check URLs

Check multiple URLs at once to see which ones have been processed.


In [ ]:
def batch_check_urls(urls: list) -> dict:
    """
    Check multiple URLs at once.
    
    Args:
        urls: List of Reddit post URLs
        
    Returns:
        Dictionary with 'seen' and 'unseen' lists
    """
    seen = []
    unseen = []
    
    for url in urls:
        result = check_url_seen(url)
        if result.get("seen"):
            seen.append({
                "url": url,
                "post_id": result.get("post_id"),
                "title": result.get("title"),
                "era": result.get("era")
            })
        else:
            unseen.append({
                "url": url,
                "post_id": result.get("post_id")
            })
    
    return {"seen": seen, "unseen": unseen}

# Example: Check all the posts we've been working with
urls_to_check = [
    "https://www.reddit.com/r/TaylorSwiftPictures/comments/1puk3m7/taylor_swift_in_golden_glam/",
    "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pkvibc/on_the_couch_in_the_end_of_an_era/",
    "https://www.reddit.com/r/TaylorSwiftPictures/comments/1oytjju/taylors_award_show_performance_outfits_through/",
]

results = batch_check_urls(urls_to_check)

print(f"URLs Checked: {len(urls_to_check)}")
print(f"Already Seen: {len(results['seen'])}")
print(f"Not Seen: {len(results['unseen'])}")
print("\n" + "=" * 80)

if results['seen']:
    print("\nAlready Processed:")
    for item in results['seen']:
        era_info = f" [{item.get('era')}]" if item.get('era') else ""
        print(f"  ✓ {item.get('title', 'N/A')}{era_info}")
        print(f"    ID: {item.get('post_id')}")

if results['unseen']:
    print("\nNot Yet Processed:")
    for item in results['unseen']:
        print(f"  ✗ {item.get('url')}")
        print(f"    ID: {item.get('post_id')}")


## Gallery Post Details

For gallery posts, we can check how many images were extracted.


In [ ]:
# Check gallery post details
gallery_post_id = extract_post_id_from_url("https://www.reddit.com/r/TaylorSwiftPictures/comments/1oytjju/taylors_award_show_performance_outfits_through/")

gallery_query = """
MATCH (p:Post {id: $post_id})
OPTIONAL MATCH (p)-[r:HAS_IMAGE]->(img:Image)
RETURN p.id as id,
       p.title as title,
       p.url as url,
       count(img) as image_count,
       collect(img.url)[0..5] as sample_images
"""

result = neo4j.execute_read(gallery_query, parameters={"post_id": gallery_post_id})

if result:
    record = result[0]
    print(f"Gallery Post: {record.get('title', 'N/A')}")
    print(f"  Post ID: {record.get('id')}")
    print(f"  Total Images: {record.get('image_count', 0)}")
    if record.get('sample_images'):
        print(f"  Sample Image URLs:")
        for i, img_url in enumerate(record['sample_images'][:3], 1):
            print(f"    {i}. {img_url[:80]}...")
else:
    print(f"Post {gallery_post_id} not found in database")


## Crawler Session Tracking

Track when the crawler last ran and estimate how many new posts have appeared since.


In [ ]:
def get_last_crawl_session(subreddit: str = "TaylorSwiftPictures") -> dict:
    """
    Get information about the last crawl session for a subreddit.
    
    Args:
        subreddit: Subreddit name to check
        
    Returns:
        Dictionary with last_crawl_time, posts_since, estimated_requests, etc.
    """
    # Find the most recent post from this subreddit
    query = """
    MATCH (p:Post)-[:POSTED_IN]->(s:Subreddit {name: $subreddit})
    RETURN p.id as id,
           p.title as title,
           p.created_utc as created,
           p.permalink as permalink,
           p.score as score
    ORDER BY p.created_utc DESC
    LIMIT 1
    """
    
    result = neo4j.execute_read(query, parameters={"subreddit": subreddit})
    
    if not result:
        return {
            "subreddit": subreddit,
            "last_crawl_time": None,
            "posts_in_db": 0,
            "posts_since_last_crawl": None,
            "estimated_requests_needed": None,
            "message": "No posts found in database for this subreddit"
        }
    
    last_post = result[0]
    last_crawl_time = last_post.get("created")
    
    # Count total posts in DB for this subreddit
    count_query = """
    MATCH (p:Post)-[:POSTED_IN]->(s:Subreddit {name: $subreddit})
    RETURN count(p) as total_posts
    """
    count_result = neo4j.execute_read(count_query, parameters={"subreddit": subreddit})
    total_posts = count_result[0].get("total_posts", 0) if count_result else 0
    
    # Calculate time since last crawl
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    
    if last_crawl_time:
        if isinstance(last_crawl_time, str):
            from dateutil import parser
            last_crawl_dt = parser.parse(last_crawl_time)
        else:
            last_crawl_dt = last_crawl_time
        
        if last_crawl_dt.tzinfo is None:
            last_crawl_dt = last_crawl_dt.replace(tzinfo=timezone.utc)
        
        time_since = now - last_crawl_dt
        days_since = time_since.total_seconds() / 86400
        hours_since = time_since.total_seconds() / 3600
        
        # Rough estimate: TaylorSwiftPictures gets ~10-50 posts per day
        # This is a rough estimate and can be adjusted based on actual activity
        estimated_posts_per_day = 20
        estimated_posts_since = int(days_since * estimated_posts_per_day)
        
        # Estimate requests needed (each post needs 1 request, plus pagination)
        # Assuming 25 posts per page on Reddit
        pages_needed = max(1, estimated_posts_since // 25)
        estimated_requests = pages_needed + estimated_posts_since  # page requests + post requests
    else:
        days_since = None
        hours_since = None
        estimated_posts_since = None
        estimated_requests = None
    
    return {
        "subreddit": subreddit,
        "last_crawl_time": last_crawl_time,
        "last_post_id": last_post.get("id"),
        "last_post_title": last_post.get("title"),
        "days_since_last_crawl": days_since,
        "hours_since_last_crawl": hours_since,
        "posts_in_db": total_posts,
        "estimated_posts_since": estimated_posts_since,
        "estimated_requests_needed": estimated_requests,
        "message": f"Last crawl was {days_since:.1f} days ago" if days_since else "No crawl history"
    }

# Check last crawl session
session_info = get_last_crawl_session("TaylorSwiftPictures")

print("Crawler Session Status")
print("=" * 80)
print(f"Subreddit: r/{session_info['subreddit']}")
print(f"Posts in Database: {session_info['posts_in_db']}")

if session_info.get('last_crawl_time'):
    print(f"\nLast Crawl:")
    print(f"  Time: {session_info['last_crawl_time']}")
    print(f"  Post: {session_info.get('last_post_title', 'N/A')}")
    print(f"  Post ID: {session_info.get('last_post_id', 'N/A')}")
    
    if session_info.get('days_since_last_crawl'):
        print(f"\nTime Since Last Crawl:")
        print(f"  Days: {session_info['days_since_last_crawl']:.1f}")
        print(f"  Hours: {session_info['hours_since_last_crawl']:.1f}")
        
        print(f"\nEstimated Activity Since Last Crawl:")
        print(f"  Estimated New Posts: ~{session_info.get('estimated_posts_since', 'N/A')}")
        print(f"  Estimated Requests Needed: ~{session_info.get('estimated_requests_needed', 'N/A')}")
        print(f"  (Note: These are rough estimates based on typical activity)")
else:
    print(f"\n{session_info.get('message', 'No crawl history found')}")


In [ ]:
# Post 4: Taylor's Workout
post_url_4 = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pxtr93/taylors_workout/"
process_taylor_swift_post(post_url_4, era_tag=None, era_full_name=None)


## Image Storage Location

The current system stores image URLs in Neo4j, but does NOT download images to disk by default.

**Current Behavior:**
- Image URLs are stored as Image nodes in Neo4j
- Images are NOT downloaded/cached locally
- Images remain at their original Reddit/CDN URLs

**Where images COULD be stored (if you enable image downloading):**
- `/tmp/image_dedup_storage` (default for polling engine)
- `data/Pictures/` (manual downloads)
- Custom path via `IMAGE_DEDUP_STORAGE_PATH` environment variable

**To query where image URLs are stored:**


In [ ]:
# Query image URLs for Taylor Swift posts
image_query = """
MATCH (c:Creator {slug: 'taylor-swift'})<-[:ABOUT]-(p:Post)-[:HAS_IMAGE]->(img:Image)
RETURN p.id as post_id,
       p.title as post_title,
       img.url as image_url,
       p.created_utc as created
ORDER BY p.created_utc DESC
LIMIT 50
"""

result = neo4j.execute_read(image_query)

print(f"Image URLs for Taylor Swift posts (showing first 20):")
print("=" * 80)

if result:
    for i, record in enumerate(result[:20], 1):
        print(f"\n{i}. Post: {record.get('post_title', 'N/A')[:60]}")
        print(f"   Post ID: {record.get('post_id')}")
        print(f"   Image URL: {record.get('image_url', 'N/A')[:80]}...")
        print(f"   Created: {record.get('created')}")
    
    print(f"\n\nTotal images found: {len(result)}")
    print("\nNote: These are URLs stored in Neo4j, not local file paths.")
    print("Images are NOT downloaded to disk - they remain at their original URLs.")
else:
    print("No images found in database")


## Download Images (Optional)

If you want to download images locally, you can use the image hashing utility:


In [ ]:
import os
from pathlib import Path
from feed.utils.image_hash import download_and_hash_image

def download_image_to_disk(image_url: str, output_dir: str = "data/Pictures/taylor_swift") -> dict:
    """
    Download an image from URL to disk.
    
    Args:
        image_url: Image URL to download
        output_dir: Directory to save image
        
    Returns:
        Dictionary with success status, file path, and hash info
    """
    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Download and hash
    sha256, dhash, image_bytes = download_and_hash_image(image_url)
    
    if not image_bytes:
        return {"success": False, "error": "Failed to download image"}
    
    # Generate filename from URL
    from urllib.parse import urlparse
    parsed = urlparse(image_url)
    filename = os.path.basename(parsed.path)
    if not filename or '.' not in filename:
        # Use hash as filename if URL doesn't have good filename
        filename = f"{sha256[:16]}.jpg"
    
    # Save to disk
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'wb') as f:
        f.write(image_bytes)
    
    return {
        "success": True,
        "filepath": filepath,
        "sha256": sha256,
        "dhash": dhash,
        "size_bytes": len(image_bytes)
    }

# Example: Download images from a specific post
post_id = "1puk3m7"  # Golden Glam post
image_urls_query = """
MATCH (p:Post {id: $post_id})-[:HAS_IMAGE]->(img:Image)
RETURN img.url as url
"""

urls_result = neo4j.execute_read(image_urls_query, parameters={"post_id": post_id})

if urls_result:
    print(f"Found {len(urls_result)} images for post {post_id}")
    print(f"Downloading to: data/Pictures/taylor_swift/")
    print("=" * 80)
    
    for i, record in enumerate(urls_result, 1):
        url = record.get('url')
        print(f"\n{i}. Downloading: {url[:60]}...")
        result = download_image_to_disk(url)
        if result.get('success'):
            print(f"   Saved: {result.get('filepath')}")
            print(f"   SHA256: {result.get('sha256')[:16]}...")
            print(f"   Size: {result.get('size_bytes')} bytes")
        else:
            print(f"   Error: {result.get('error')}")
else:
    print(f"No images found for post {post_id}")


## Metadata Storage and Cache Hits

**Metadata Storage Location**: All Reddit post metadata is stored in **Neo4j database**, NOT in local files.

- **Post metadata**: Stored as `Post` nodes in Neo4j
- **Images**: Stored as `Image` nodes linked to posts
- **Comments**: Stored as `Comment` nodes linked to posts
- **Relationships**: Creator links, subreddit links, era tags all in Neo4j

**No local JSON/JSONL files** - Neo4j is the single source of truth for all crawler data.

**Cache Hit Strategy**: Before fetching from Reddit API, check Neo4j first. If post exists (cache hit), use cached data. Only fetch if not found (cache miss).


In [ ]:
def check_post_metadata_in_neo4j(post_id: str) -> dict:
    """
    Check if post metadata exists in Neo4j database (cache hit check).
    
    Args:
        post_id: Reddit post ID
        
    Returns:
        Dictionary with metadata if found, or None
    """
    query = """
    MATCH (p:Post {id: $post_id})
    OPTIONAL MATCH (p)-[:HAS_IMAGE]->(img:Image)
    OPTIONAL MATCH (p)-[:ABOUT]->(c:Creator)
    OPTIONAL MATCH (p)-[:POSTED_IN]->(s:Subreddit)
    OPTIONAL MATCH (p)<-[:POSTED]-(u:User)
    RETURN p.id as id,
           p.title as title,
           p.url as url,
           p.permalink as permalink,
           p.selftext as selftext,
           p.created_utc as created_utc,
           p.score as score,
           p.num_comments as num_comments,
           p.era as era,
           p.era_full_name as era_full_name,
           p.subreddit as subreddit,
           p.updated_at as updated_at,
           count(img) as image_count,
           collect(img.url)[0..5] as image_urls,
           c.slug as creator_slug,
           c.name as creator_name,
           s.name as subreddit_name,
           u.username as author
    """
    
    result = neo4j.execute_read(query, parameters={"post_id": post_id})
    
    if result:
        record = result[0]
        return {
            "cache_hit": True,
            "post_id": record.get("id"),
            "title": record.get("title"),
            "url": record.get("url"),
            "permalink": record.get("permalink"),
            "created_utc": record.get("created_utc"),
            "score": record.get("score"),
            "num_comments": record.get("num_comments"),
            "era": record.get("era"),
            "era_full_name": record.get("era_full_name"),
            "subreddit": record.get("subreddit_name"),
            "author": record.get("author"),
            "image_count": record.get("image_count", 0),
            "image_urls": [url for url in record.get("image_urls", []) if url],
            "creator_slug": record.get("creator_slug"),
            "creator_name": record.get("creator_name"),
            "updated_at": record.get("updated_at"),
            "has_content": bool(record.get("selftext")),
            "content_length": len(record.get("selftext", ""))
        }
    
    return {"cache_hit": False, "post_id": post_id}

# Check the new post
post_url = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pxtr93/taylors_workout/"
post_id = extract_post_id_from_url(post_url)

print("Cache Hit Check")
print("=" * 80)
print(f"Post ID: {post_id}")
print(f"URL: {post_url}")
print()

metadata = check_post_metadata_in_neo4j(post_id)

if metadata.get("cache_hit"):
    print("CACHE HIT: Post found in Neo4j database")
    print("=" * 80)
    print(f"Title: {metadata.get('title', 'N/A')}")
    print(f"Subreddit: r/{metadata.get('subreddit', 'N/A')}")
    print(f"Author: u/{metadata.get('author', 'N/A')}")
    print(f"Created: {metadata.get('created_utc', 'N/A')}")
    print(f"Score: {metadata.get('score', 'N/A')}")
    print(f"Comments: {metadata.get('num_comments', 'N/A')}")
    print(f"Era: {metadata.get('era', 'N/A')}")
    print(f"Images: {metadata.get('image_count', 0)}")
    print(f"Creator: {metadata.get('creator_name', 'N/A')} (slug: {metadata.get('creator_slug', 'N/A')})")
    print(f"Updated: {metadata.get('updated_at', 'N/A')}")
    print(f"Has Content: {metadata.get('has_content', False)} ({metadata.get('content_length', 0)} chars)")
    
    if metadata.get("image_urls"):
        print(f"\nImage URLs ({len(metadata['image_urls'])}):")
        for i, img_url in enumerate(metadata['image_urls'][:3], 1):
            print(f"  {i}. {img_url[:70]}...")
else:
    print("CACHE MISS: Post not found in Neo4j database")
    print("Post needs to be fetched and stored")
    print(f"Use process_taylor_swift_post() to fetch and store it")


## Process Post with Cache Check

Process a post with cache-hit checking - only fetch if not already in database.


In [ ]:
def process_with_cache_check(post_url: str, era_tag: str = None, era_full_name: str = None) -> dict:
    """
    Process a post with cache-hit checking. Only fetches from Reddit if not in database.
    
    Args:
        post_url: Reddit post URL
        era_tag: Optional era tag
        era_full_name: Optional full era name
        
    Returns:
        Dictionary with 'cache_hit' (bool), 'processed' (bool), and metadata
    """
    post_id = extract_post_id_from_url(post_url)
    
    # Check cache first
    metadata = check_post_metadata_in_neo4j(post_id)
    
    if metadata.get("cache_hit"):
        print(f"CACHE HIT: Post {post_id} already in database")
        print(f"  Title: {metadata.get('title', 'N/A')}")
        print(f"  Era: {metadata.get('era', 'N/A')}")
        
        # Update era if provided and different
        if era_tag and metadata.get("era") != era_tag:
            print(f"  Updating era from '{metadata.get('era')}' to '{era_tag}'")
            update_era_query = """
            MATCH (p:Post {id: $post_id})
            SET p.era = $era,
                p.era_full_name = $era_full_name,
                p.updated_at = datetime()
            RETURN p.era as era
            """
            neo4j.execute_write(
                update_era_query,
                parameters={
                    "post_id": post_id,
                    "era": era_tag,
                    "era_full_name": era_full_name or era_tag
                }
            )
            print(f"  Era updated successfully")
        
        return {
            "cache_hit": True,
            "processed": False,
            "metadata": metadata
        }
    
    # Cache miss - process the post
    print(f"CACHE MISS: Post {post_id} not in database, fetching from Reddit...")
    process_taylor_swift_post(post_url, era_tag=era_tag, era_full_name=era_full_name)
    
    # Re-check to get updated metadata
    updated_metadata = check_post_metadata_in_neo4j(post_id)
    
    return {
        "cache_hit": False,
        "processed": True,
        "metadata": updated_metadata
    }

# Process the workout post with cache checking
post_url = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pxtr93/taylors_workout/"
result = process_with_cache_check(post_url, era_tag=None, era_full_name=None)

print("\n" + "=" * 80)
print("Processing Result:")
print(f"  Cache Hit: {result.get('cache_hit')}")
print(f"  Processed: {result.get('processed')}")
if result.get('metadata'):
    print(f"  Post ID: {result['metadata'].get('post_id')}")
    print(f"  Title: {result['metadata'].get('title', 'N/A')}")


In [ ]:
# Post 2: End of an Era (era unknown - can be updated later)
post_url_2 = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pkvibc/on_the_couch_in_the_end_of_an_era/"
process_taylor_swift_post(post_url_2, era_tag=None, era_full_name=None)


## Query Posts by Subreddit

Query all Taylor Swift posts from a specific subreddit.


In [ ]:
# Query all Taylor Swift posts from TaylorSwiftPictures subreddit
query = """
MATCH (c:Creator {slug: 'taylor-swift'})-[:HAS_SOURCE]->(s:Subreddit {name: 'TaylorSwiftPictures'})
MATCH (p:Post)-[:POSTED_IN]->(s)
MATCH (p)-[:ABOUT]->(c)
RETURN p.id as id, p.title as title, p.created_utc as created, p.era as era, p.era_full_name as era_full_name
ORDER BY p.created_utc DESC
LIMIT 20
"""

result = neo4j.execute_read(query)
print(f"Found {len(result)} Taylor Swift posts from r/TaylorSwiftPictures:")
print("=" * 80)
for record in result:
    era_info = f" [{record.get('era', 'N/A')}]" if record.get('era') else ""
    print(f"\n{record.get('title', 'N/A')}{era_info}")
    print(f"  ID: {record.get('id', 'N/A')}")
    print(f"  Created: {record.get('created', 'N/A')}")


## Query All Subreddits Tracking Taylor Swift

See all the Reddit subreddits that are linked as sources for Taylor Swift content.


## Blog Feed Tracker

This section demonstrates tracking blog posts from RSS feeds, enriching feed content with full post content, and filtering by specific entities.


In [ ]:
from feed.platforms.blog import BlogAdapter
from feed.storage.thread_storage import store_blog_post
from feed.services.creator_service import CreatorService

# Initialize blog adapter
blog = BlogAdapter()


### Process Blog Feed with Entity Filtering

This function processes a blog RSS feed, enriches posts with full content, and filters by specific entities (e.g., "princess lexie" but not "princess lexi").


In [ ]:
def process_blog_feed(feed_url: str, entity_filter: str = None, scrape_content: bool = True, limit: int = 50, creator_slug: str = None):
    """
    Process a blog RSS feed, enrich with full content, and optionally filter by entity.
    
    Args:
        feed_url: RSS feed URL
        entity_filter: Optional entity name to filter posts (e.g., "princess lexie")
        scrape_content: Whether to scrape full content from post URLs
        limit: Maximum number of posts to process
        creator_slug: Optional creator slug to link posts to Creator entity
    """
    print(f"Processing blog feed: {feed_url}")
    if entity_filter:
        print(f"Entity filter: {entity_filter}")
    print("=" * 80)
    
    # Fetch posts from RSS feed
    posts, next_token = blog.fetch_posts(
        source=feed_url,
        limit=limit,
        entity_filter=entity_filter,
        scrape_content=scrape_content,
    )
    
    print(f"Found {len(posts)} posts")
    
    if not posts:
        print("No posts found matching criteria")
        return
    
    # Process each post
    for i, post in enumerate(posts, 1):
        print(f"\n[{i}/{len(posts)}] Processing: {post.title}")
        print(f"  URL: {post.url}")
        print(f"  Created: {post.created_utc}")
        
        # Check if post already exists
        check_query = """
        MATCH (p:Post {id: $post_id})
        RETURN p.id as id, p.title as title, p.url as url, p.entity_matched as entity
        LIMIT 1
        """
        result = neo4j.execute_read(check_query, parameters={"post_id": post.id})
        
        if result:
            record = result[0]
            print(f"  Already exists in database (entity: {record.get('entity', 'N/A')})")
            continue
        
        # Get metadata from blog adapter
        metadata = blog.get_post_metadata(post.id)
        images = metadata.get('images', []) if metadata else []
        entity_matched = metadata.get('entity_matched') if metadata else None
        
        print(f"  Content length: {len(post.selftext)} chars")
        print(f"  Images found: {len(images)}")
        if entity_matched:
            print(f"  Entity matched: {entity_matched}")
        
        # Store in Neo4j
        try:
            # Determine creator slug: use provided parameter, or derive from entity_matched
            post_creator_slug = creator_slug
            if not post_creator_slug and entity_matched:
                # Convert entity name to slug (e.g., "princess lexie" -> "princess-lexie")
                post_creator_slug = entity_matched.lower().replace(' ', '-')
            
            store_blog_post(neo4j, post, images=images, entity_matched=entity_matched, creator_slug=post_creator_slug)
            print(f"  Successfully stored post {post.id} in database!")
            
            # Link to blog source (using subreddit field as blog name)
            link_blog_query = """
            MATCH (p:Post {id: $post_id})
            MATCH (s:Subreddit {name: $blog_name})
            MERGE (p)-[:POSTED_IN]->(s)
            """
            neo4j.execute_write(
                link_blog_query,
                parameters={
                    "post_id": post.id,
                    "blog_name": post.subreddit,
                }
            )
            print(f"  Linked post to blog source: {post.subreddit}")
            
            if creator_slug:
                print(f"  Linked post to Creator entity: {creator_slug}")
            
        except Exception as e:
            print(f"  Error storing post: {e}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "=" * 80)
    print(f"Processing complete: {len(posts)} posts processed")


In [ ]:
# Process blog feed with entity filtering and link to Creator entity
feed_url = "https://femdom-pov.me/feed/"
entity_filter = "princess lexie"  # Exact match - will match "princess lexie" but not "princess lexi"

process_blog_feed(
    feed_url=feed_url,
    entity_filter=entity_filter,
    scrape_content=True,  # Enrich with full post content
    limit=50,
    creator_slug="princess-lexie"  # Link posts to Princess Lexie Creator entity
)


### Query Blog Posts by Entity

Query all blog posts that matched a specific entity.


In [ ]:
# Query posts matching "princess lexie"
query = """
MATCH (p:Post)
WHERE p.entity_matched = $entity
RETURN p.id as id, 
       p.title as title, 
       p.url as url,
       p.created_utc as created,
       p.entity_matched as entity,
       size([(p)-[:HAS_IMAGE]->(img:Image) | img]) as image_count
ORDER BY p.created_utc DESC
LIMIT 20
"""

result = neo4j.execute_read(query, parameters={"entity": "princess lexie"})
print(f"Found {len(result)} posts matching 'princess lexie':")
print("=" * 80)
for record in result:
    print(f"\n{record.get('title', 'N/A')}")
    print(f"  ID: {record.get('id', 'N/A')}")
    print(f"  URL: {record.get('url', 'N/A')}")
    print(f"  Created: {record.get('created', 'N/A')}")
    print(f"  Images: {record.get('image_count', 0)}")


### View Full Post Content

View the enriched content of a specific blog post.


### Register Multiple Sources for a Creator

Link multiple platforms/sources to a Creator entity so you can filter all their content across platforms.


In [ ]:
def register_creator_sources(creator_name: str, creator_slug: str, sources: list):
    """
    Register multiple sources/platforms for a creator.
    
    Args:
        creator_name: Creator display name (e.g., "Princess Lexie")
        creator_slug: Creator slug (e.g., "princess-lexie")
        sources: List of dicts with 'name', 'type', and optional 'url'
                 Example: [
                     {'name': 'PrincessLexie', 'type': 'reddit', 'url': 'https://www.reddit.com/r/PrincessLexie/'},
                     {'name': 'femdom-pov', 'type': 'blog', 'url': 'https://femdom-pov.me/feed/'},
                     {'name': 'PrincessLexieXO', 'type': 'twitter', 'url': 'https://x.com/PrincessLexieXO/'},
                     {'name': '405/Princess-Lexie', 'type': 'iwantclips', 'url': 'https://iwantclips.com/store/405/Princess-Lexie'},
                 ]
    """
    creator_service = CreatorService()
    
    # Create or get creator
    creator = creator_service.get_or_create_creator(
        name=creator_name,
        slug=creator_slug
    )
    print(f"Creator: {creator.get('name', 'N/A')} (slug: {creator.get('slug', 'N/A')})")
    print("=" * 80)
    
    # Register each source
    for source in sources:
        source_name = source.get('name')
        source_type = source.get('type')
        source_url = source.get('url')
        
        print(f"\nRegistering source: {source_name} ({source_type})")
        if source_url:
            print(f"  URL: {source_url}")
        
        success = creator_service.link_source_to_creator(
            creator_slug=creator_slug,
            source_name=source_name,
            source_type=source_type,
            source_url=source_url,
        )
        
        if success:
            print(f"  Successfully linked {source_type} source '{source_name}' to {creator_name}")
        else:
            print(f"  Failed to link {source_type} source '{source_name}'")
    
    print("\n" + "=" * 80)
    print(f"Registration complete for {creator_name}")

# Example: Register all sources for Princess Lexie
princess_lexie_sources = [
    {
        'name': 'PrincessLexie',
        'type': 'reddit',
        'url': 'https://www.reddit.com/r/PrincessLexie/'
    },
    {
        'name': 'femdom-pov',
        'type': 'blog',
        'url': 'https://femdom-pov.me/feed/'
    },
    {
        'name': 'PrincessLexieXO',
        'type': 'twitter',
        'url': 'https://x.com/PrincessLexieXO/'
    },
    {
        'name': '405/Princess-Lexie',
        'type': 'iwantclips',
        'url': 'https://iwantclips.com/store/405/Princess-Lexie'
    },
]

register_creator_sources(
    creator_name="Princess Lexie",
    creator_slug="princess-lexie",
    sources=princess_lexie_sources
)


In [ ]:
# Query all content for Princess Lexie across all platforms
creator_slug = "princess-lexie"

query = """
MATCH (c:Creator {slug: $creator_slug})-[:HAS_SOURCE]->(s:Subreddit)
MATCH (p:Post)-[:POSTED_IN]->(s)
OPTIONAL MATCH (p)-[:ABOUT]->(c)
WITH p, s, c
WHERE (p)-[:ABOUT]->(c) OR (p)-[:POSTED_IN]->(s)
RETURN p.id as id,
       p.title as title,
       p.url as url,
       p.created_utc as created,
       s.name as source_name,
       s.source_type as source_type,
       p.entity_matched as entity_matched,
       size([(p)-[:HAS_IMAGE]->(img:Image) | img]) as image_count
ORDER BY p.created_utc DESC
LIMIT 50
"""

result = neo4j.execute_read(query, parameters={"creator_slug": creator_slug})
print(f"Found {len(result)} posts for Princess Lexie across all platforms:")
print("=" * 80)

# Group by source type
by_source = {}
for record in result:
    source_type = record.get('source_type', 'unknown')
    if source_type not in by_source:
        by_source[source_type] = []
    by_source[source_type].append(record)

for source_type, posts in by_source.items():
    print(f"\n{source_type.upper()} ({len(posts)} posts):")
    for record in posts[:5]:  # Show first 5 per source
        entity_info = f" [entity: {record.get('entity_matched', 'N/A')}]" if record.get('entity_matched') else ""
        print(f"  - {record.get('title', 'N/A')[:60]}...")
        print(f"    Source: {record.get('source_name', 'N/A')} | Images: {record.get('image_count', 0)}{entity_info}")
    if len(posts) > 5:
        print(f"  ... and {len(posts) - 5} more")


### View All Registered Sources for a Creator

See all the platforms/sources linked to a creator.


In [ ]:
# View all sources registered for a creator
creator_slug = "princess-lexie"

query = """
MATCH (c:Creator {slug: $creator_slug})-[r:HAS_SOURCE]->(s:Subreddit)
RETURN c.name as creator_name,
       c.slug as creator_slug,
       s.name as source_name,
       s.source_type as source_type,
       r.source_url as source_url,
       r.discovered_at as discovered_at,
       count { (s)<-[:POSTED_IN]-(p:Post) } as post_count
ORDER BY s.source_type, s.name
"""

result = neo4j.execute_read(query, parameters={"creator_slug": creator_slug})

if result:
    record = result[0]
    print(f"Creator: {record.get('creator_name', 'N/A')} (slug: {record.get('creator_slug', 'N/A')})")
    print("=" * 80)
    print("\nRegistered Sources:")
    
    for record in result:
        source_type = record.get('source_type', 'unknown')
        source_name = record.get('source_name', 'N/A')
        source_url = record.get('source_url', 'N/A')
        post_count = record.get('post_count', 0)
        discovered = record.get('discovered_at', 'N/A')
        
        print(f"\n  {source_type.upper()}: {source_name}")
        if source_url and source_url != 'N/A':
            print(f"    URL: {source_url}")
        print(f"    Posts: {post_count}")
        print(f"    Discovered: {discovered}")
else:
    print(f"No sources found for creator '{creator_slug}'")
    print("Register sources using register_creator_sources()")


In [ ]:
# Example: View full content of a specific post
example_post_url = "https://femdom-pov.me/princess-lexie-height-comparison-joi/"

# Extract post ID from URL
from urllib.parse import urlparse
parsed = urlparse(example_post_url)
post_id = parsed.path.strip('/').split('/')[-1].replace('.html', '').replace('.php', '')

content_query = """
MATCH (p:Post {id: $post_id})
OPTIONAL MATCH (p)-[:HAS_IMAGE]->(img:Image)
RETURN p.id as id,
       p.title as title,
       p.url as url,
       p.selftext as content,
       p.entity_matched as entity,
       collect(img.url)[0..10] as images
"""

result = neo4j.execute_read(content_query, parameters={"post_id": post_id})

if result:
    record = result[0]
    print(f"Post: {record.get('title', 'N/A')}")
    print(f"URL: {record.get('url', 'N/A')}")
    print(f"Entity: {record.get('entity', 'N/A')}")
    print(f"\nFull Content ({len(record.get('content', ''))} chars):")
    print("=" * 80)
    content = record.get('content', '')
    if content:
        # Show first 1000 chars
        print(content[:1000])
        if len(content) > 1000:
            print(f"\n... ({len(content) - 1000} more characters)")
    else:
        print("(No content stored)")
    
    images = record.get('images', [])
    if images:
        print(f"\nImages ({len(images)}):")
        for i, img_url in enumerate(images[:5], 1):
            print(f"  {i}. {img_url}")
else:
    print(f"Post {post_id} not found in database")
    print("You can process it using process_blog_feed()")


## Intelligent Feed Monitoring with Post Count Statistics

This section demonstrates how to use subreddit statistics to make our Reddit data feed more intelligent. By tracking post counts over time, we can determine optimal crawling policies for each subreddit.

### Key Concepts

1. **Post Count Metrics**: Track monthly/yearly post counts to understand subreddit activity
2. **Post Velocity**: Calculate average posts per day to determine how frequently to crawl
3. **Crawler Policy**: Automatically adjust crawl frequency, delays, and page limits based on activity

### Methods for Getting Post Counts

- **API-style queries**: Use Reddit's JSON API with time range filters (after/before timestamps)
- **Rolling monthly metrics**: Track post counts for the last N months
- **Yearly aggregations**: Sum monthly counts or query year-wide ranges
- **All-time estimates**: Approximate by querying from Reddit's founding (2005) to now

Note: Reddit doesn't expose native "all-time" post counts, but we can approximate using time-range queries. For more complete historical coverage, consider using public Reddit datasets (e.g., BigQuery mirrors).


In [ ]:
from feed.services.subreddit_stats import SubredditStatsService
from feed.services.crawler_policy import CrawlerPolicy

# Initialize services
stats_service = SubredditStatsService(neo4j=neo4j)
policy_service = CrawlerPolicy(neo4j=neo4j, stats_service=stats_service)

print("Subreddit Statistics and Crawler Policy Services initialized")


### Get Monthly Post Counts

Fetch post counts for specific months to understand subreddit activity patterns.


In [ ]:
# Get monthly post counts for a subreddit
subreddit = "TaylorSwift"

# Get counts for recent months
from datetime import datetime
today = datetime.utcnow()

print(f"Fetching monthly post counts for r/{subreddit}...")
print("=" * 80)

for i in range(3):  # Last 3 months
    target_date = today.replace(day=1) - timedelta(days=30 * i)
    year = target_date.year
    month = target_date.month
    
    count = stats_service.get_monthly_post_count(subreddit, year, month)
    if count is not None:
        print(f"{year}-{month:02d}: {count} posts")
    else:
        print(f"{year}-{month:02d}: Error fetching count")


at 

In [ ]:
# Calculate post velocity (average posts per day)
subreddit = "TaylorSwift"
velocity = stats_service.calculate_post_velocity(subreddit, months=3)

if velocity:
    print(f"r/{subreddit} post velocity: {velocity:.2f} posts/day")
    print(f"  This means approximately {velocity * 30:.0f} posts per month")
    
    if velocity >= 50:
        print("  Classification: VERY HIGH activity - crawl every 1-2 hours")
    elif velocity >= 20:
        print("  Classification: HIGH activity - crawl every 3-6 hours")
    elif velocity >= 5:
        print("  Classification: MEDIUM activity - crawl every 12-24 hours")
    elif velocity >= 1:
        print("  Classification: LOW activity - crawl every 1-2 days")
    else:
        print("  Classification: VERY LOW activity - crawl every 3-7 days")
else:
    print(f"Could not calculate velocity for r/{subreddit}")


### Get Intelligent Crawler Policy

The crawler policy service automatically determines optimal crawling parameters based on post velocity:
- **Crawl frequency**: How often to check the subreddit
- **Request delays**: How long to wait between requests within a crawl
- **Page limits**: How many pages to fetch per crawl


In [ ]:
# Get intelligent crawler policy for a subreddit
subreddit = "TaylorSwift"
policy = policy_service.get_subreddit_policy(subreddit, months=3)

print(f"Crawler Policy for r/{subreddit}:")
print("=" * 80)
print(f"Frequency Level: {policy['frequency']}")
print(f"Post Velocity: {policy['post_velocity']:.2f} posts/day" if policy['post_velocity'] else "Post Velocity: Unknown")
print(f"\nRecommended Schedule:")
print(f"  {policy['recommended_schedule']}")
print(f"\nCrawl Delays (between crawls):")
print(f"  Min: {policy['crawl_delay_seconds']['min'] / 3600:.1f} hours")
print(f"  Max: {policy['crawl_delay_seconds']['max'] / 3600:.1f} hours")
print(f"\nRequest Delays (within crawl):")
print(f"  Min: {policy['request_delay_seconds']['min']:.1f} seconds")
print(f"  Max: {policy['request_delay_seconds']['max']:.1f} seconds")
print(f"\nMax Pages per Crawl: {policy['max_pages_per_crawl']}")


 the

In [ ]:
# Store statistics for a subreddit
subreddit = "TaylorSwift"

# Get rolling monthly counts
print(f"Fetching statistics for r/{subreddit}...")
monthly_counts = stats_service.get_rolling_monthly_counts(subreddit, months=6)
velocity = stats_service.calculate_post_velocity(subreddit, months=3)

# Prepare stats dictionary
stats = {
    "monthly_counts": monthly_counts,
    "post_velocity": velocity,
    "calculated_at": datetime.utcnow().isoformat(),
}

# Store in Neo4j
success = stats_service.store_subreddit_stats(subreddit, stats)
if success:
    print(f"Successfully stored statistics for r/{subreddit}")
    print(f"  Monthly counts: {len(monthly_counts)} months")
    print(f"  Post velocity: {velocity:.2f} posts/day" if velocity else "  Post velocity: Unknown")
else:
    print(f"Failed to store statistics for r/{subreddit}")

# Store policy
policy = policy_service.get_subreddit_policy(subreddit, months=3)
policy_success = policy_service.store_policy(subreddit, policy)
if policy_success:
    print(f"Successfully stored crawler policy for r/{subreddit}")
else:
    print(f"Failed to store crawler policy for r/{subreddit}")


### Query Stored Statistics

Retrieve previously stored statistics and policies from Neo4j.


In [ ]:
# Query stored statistics
subreddit = "TaylorSwift"
stored_stats = stats_service.get_subreddit_stats(subreddit)

if stored_stats and stored_stats.get('stats'):
    stats = stored_stats['stats']
    print(f"Stored Statistics for r/{subreddit}:")
    print("=" * 80)
    print(f"Updated at: {stored_stats.get('updated_at', 'N/A')}")
    
    if 'monthly_counts' in stats:
        print(f"\nMonthly Post Counts:")
        for month, count in sorted(stats['monthly_counts'].items()):
            print(f"  {month}: {count} posts")
    
    if 'post_velocity' in stats and stats['post_velocity']:
        print(f"\nPost Velocity: {stats['post_velocity']:.2f} posts/day")
else:
    print(f"No stored statistics found for r/{subreddit}")
    print("Run the previous cell to fetch and store statistics")


f the 

In [ ]:
# Get policies for multiple Taylor Swift subreddits
taylor_subreddits = [
    "TaylorSwift",
    "TaylorSwiftPictures",
    "TaylorSwiftCandids",
]

print("Comparing crawler policies across Taylor Swift subreddits:")
print("=" * 80)

policies = policy_service.get_all_subreddit_policies(taylor_subreddits, months=3)

for subreddit, policy in policies.items():
    velocity = policy.get('post_velocity')
    frequency = policy.get('frequency', 'unknown')
    
    print(f"\nr/{subreddit}:")
    if velocity:
        print(f"  Velocity: {velocity:.2f} posts/day")
        print(f"  Frequency: {frequency}")
        print(f"  Schedule: {policy.get('recommended_schedule', 'N/A')}")
    else:
        print(f"  Velocity: Unknown (may need to fetch statistics first)")


## 4chan Thread Monitor

This section demonstrates monitoring 4chan threads for new posts and images, similar to the Reddit tracker functionality.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent))

from parse_4chan_thread import FourChanThreadParser
from feed.storage.neo4j_connection import get_connection

# Connect to Neo4j
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}")


### Monitor a 4chan Thread

Monitor a specific 4chan thread for updates. The monitor will check the thread at regular intervals and store new posts and images in Neo4j.


In [ ]:
def check_4chan_thread(thread_url: str):
    """
    Check a 4chan thread for updates and store in Neo4j.
    
    Args:
        thread_url: 4chan thread URL (e.g., https://boards.4chan.org/b/thread/944146857)
    """
    parser = FourChanThreadParser(download_images=True)
    
    # Parse URL
    url_info = parser.parse_url(thread_url)
    board = url_info["board"]
    thread_id = url_info["thread_id"]
    
    print(f"Checking thread: /{board}/{thread_id}")
    print(f"URL: {thread_url}")
    print("=" * 80)
    
    # Check if thread exists
    existing = parser.check_thread_exists(neo4j, board, thread_id)
    if existing:
        print(f"Thread already exists in database")
        print(f"  Last crawled: {existing.get('last_crawled_at')}")
        print(f"  Post count: {existing.get('post_count')}")
        print(f"  Content hash: {existing.get('content_hash', 'N/A')[:16]}...")
    
    try:
        # Fetch and parse thread
        html_text, html_bytes = parser.fetch_thread_html(thread_url)
        thread_data = parser.parse_thread_html(html_text, thread_url)
        
        # Cache HTML
        html_path = parser.cache_html(thread_url, html_bytes)
        
        # Compute content hash
        content_hash = parser.compute_content_hash(thread_data)
        
        # Check if content changed
        changed = True
        if existing:
            existing_hash = existing.get("content_hash")
            if existing_hash == content_hash:
                changed = False
                print("  -> No changes detected")
            else:
                print("  -> Changes detected!")
                old_post_count = existing.get("post_count", 0)
                new_post_count = thread_data["post_count"]
                if new_post_count > old_post_count:
                    print(f"  -> New posts: {new_post_count - old_post_count}")
        else:
            print("  -> First time seeing this thread")
        
        # Count images
        image_count = sum(1 for p in thread_data.get("posts", []) if p.get("image_url"))
        print(f"  -> Posts: {thread_data['post_count']} (with images: {image_count})")
        
        # Store in database
        parser.store_thread(neo4j, thread_data, html_path)
        
        # Check for linked threads
        linked_threads = []
        for post_data in thread_data.get("posts", []):
            linked_thread_ids = post_data.get("linked_thread_ids", [])
            for linked_thread_id in linked_thread_ids:
                if linked_thread_id != thread_id:
                    linked_threads.append(linked_thread_id)
        
        if linked_threads:
            print(f"  -> Found {len(linked_threads)} linked thread(s): {linked_threads}")
        
        print(f"\nThread stored successfully!")
        return {
            "success": True,
            "changed": changed,
            "post_count": thread_data["post_count"],
            "image_count": image_count,
            "linked_threads": linked_threads,
        }
        
    except Exception as e:
        print(f"  -> ERROR: {e}")
        import traceback
        traceback.print_exc()
        return {
            "success": False,
            "error": str(e),
        }

# Check the specified thread
thread_url = "https://boards.4chan.org/b/thread/944153883"
result = check_4chan_thread(thread_url)


### Query Monitored 4chan Threads

Query all 4chan threads that have been monitored and stored in the database.


In [ ]:
# Query all monitored 4chan threads
query = """
MATCH (t:Thread)
WHERE t.board IS NOT NULL AND t.thread_id IS NOT NULL
RETURN t.board as board,
       t.thread_id as thread_id,
       t.title as title,
       t.post_count as post_count,
       t.last_crawled_at as last_crawled_at,
       t.created_at as created_at
ORDER BY t.last_crawled_at DESC
LIMIT 20
"""

result = neo4j.execute_read(query)
print(f"Found {len(result)} monitored threads:")
print("=" * 80)
for record in result:
    print(f"\n/{record.get('board', 'N/A')}/{record.get('thread_id', 'N/A')}")
    print(f"  Title: {record.get('title', 'N/A')[:60]}")
    print(f"  Posts: {record.get('post_count', 0)}")
    print(f"  Last crawled: {record.get('last_crawled_at', 'N/A')}")
    print(f"  Created: {record.get('created_at', 'N/A')}")


### Refresh Thread Catalog with Keyword Filter

Refresh the catalog and filter threads by keywords (e.g., "girls", "irl", "feet").


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.platforms.fourchan import FourChanAdapter
from datetime import datetime

# Initialize 4chan adapter with keyword filtering
# Keywords: "girls", "irl", or "feet" (case-insensitive, whole-word matching)
keywords = ["girls", "irl", "feet"]
board = "b"  # Random image board

print(f"Refreshing thread catalog for /{board}/ with keywords: {keywords}")
print("=" * 80)
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

fourchan = FourChanAdapter(
    delay_min=2.0,
    delay_max=5.0,
    keywords=keywords,
    mock=False
)

# Fetch threads from catalog
threads = fourchan.fetch_threads(board)

print(f"Found {len(threads)} threads matching keywords: {keywords}")
print("=" * 80)

if threads:
    print(f"\nThreads found:")
    for i, thread in enumerate(threads[:20], 1):  # Show first 20
        thread_id = thread.get("no")
        subject = thread.get("sub", "") or "No subject"
        replies = thread.get("replies", 0)
        images = thread.get("images", 0)
        
        # Extract text from HTML if present
        if subject and "<" in subject:
            import re
            subject = re.sub(r'<[^>]+>', '', subject)
        
        print(f"\n{i}. /{board}/{thread_id}")
        print(f"   Subject: {subject[:80]}")
        print(f"   Replies: {replies} | Images: {images}")
    
    if len(threads) > 20:
        print(f"\n... and {len(threads) - 20} more threads")
else:
    print("No threads found matching the keywords")

print(f"\nTotal threads in catalog matching keywords: {len(threads)}")


### Check and Refresh Tracked Threads

Query which threads we're currently tracking that match our keywords, then refresh them.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))
sys.path.insert(0, str(Path().absolute().parent.parent))

from feed.storage.neo4j_connection import get_connection
from parse_4chan_thread import FourChanThreadParser
from feed.platforms.fourchan import FourChanAdapter
from datetime import datetime
import re

# Connect to Neo4j
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}")

# Keywords we're tracking
keywords = ["girls", "irl", "feet"]
board = "b"

print(f"Checking tracked threads on /{board}/ matching keywords: {keywords}")
print("=" * 80)
print()

# Query threads matching keywords
# Check if title contains any of our keywords (case-insensitive)
keyword_pattern = "|".join([re.escape(k) for k in keywords])
query = f"""
MATCH (t:Thread)
WHERE t.board = $board
  AND (
    toLower(t.title) CONTAINS 'girls' OR
    toLower(t.title) CONTAINS 'irl' OR
    toLower(t.title) CONTAINS 'feet'
  )
RETURN t.board as board,
       t.thread_id as thread_id,
       t.title as title,
       t.post_count as post_count,
       t.last_crawled_at as last_crawled_at,
       t.created_at as created_at,
       t.url as url
ORDER BY t.last_crawled_at DESC NULLS LAST
"""

result = neo4j.execute_read(query, parameters={"board": board})

print(f"Found {len(result)} tracked threads matching keywords")
print("=" * 80)

if not result:
    print("No threads found in database matching keywords.")
    print("Threads may need to be added from catalog search first.")
else:
    print(f"\nTracked Threads:")
    for i, record in enumerate(result, 1):
        thread_id = record.get('thread_id')
        title = record.get('title', 'No title')[:60]
        post_count = record.get('post_count', 0)
        last_crawled = record.get('last_crawled_at', 'Never')
        url = record.get('url')
        
        print(f"\n{i}. /{board}/{thread_id}")
        print(f"   Title: {title}")
        print(f"   Posts: {post_count}")
        print(f"   Last crawled: {last_crawled}")
        if url:
            print(f"   URL: {url}")

print("\n" + "=" * 80)
print("Refreshing threads...")
print("=" * 80)

# Initialize parser for refreshing threads
parser = FourChanThreadParser(download_images=True)

# Refresh each thread
refreshed_count = 0
error_count = 0

for i, record in enumerate(result, 1):
    thread_id = record.get('thread_id')
    title = record.get('title', 'No title')[:60]
    
    print(f"\n[{i}/{len(result)}] Refreshing /{board}/{thread_id}")
    print(f"  Title: {title}")
    
    # Build thread URL
    thread_url = f"https://boards.4chan.org/{board}/thread/{thread_id}"
    
    try:
        # Check if thread exists in database
        existing = parser.check_thread_exists(neo4j, board, thread_id)
        if existing:
            print(f"  Current post count: {existing.get('post_count', 0)}")
            print(f"  Last crawled: {existing.get('last_crawled_at', 'Never')}")
        
        # Fetch and parse thread
        html_text, html_bytes = parser.fetch_thread_html(thread_url)
        thread_data = parser.parse_thread_html(html_text, thread_url)
        
        # Cache HTML
        html_path = parser.cache_html(thread_url, html_bytes)
        
        # Compute content hash
        content_hash = parser.compute_content_hash(thread_data)
        
        # Check if content changed
        changed = True
        if existing:
            existing_hash = existing.get("content_hash")
            if existing_hash == content_hash:
                changed = False
                print(f"  -> No changes detected")
            else:
                print(f"  -> Changes detected!")
                old_post_count = existing.get("post_count", 0)
                new_post_count = thread_data["post_count"]
                if new_post_count > old_post_count:
                    print(f"  -> New posts: {new_post_count - old_post_count}")
        
        # Count images
        image_count = sum(1 for p in thread_data.get("posts", []) if p.get("image_url"))
        print(f"  -> Posts: {thread_data['post_count']} (with images: {image_count})")
        
        # Store in database
        parser.store_thread(neo4j, thread_data, html_path)
        
        print(f"  -> Successfully refreshed!")
        refreshed_count += 1
        
    except Exception as e:
        print(f"  -> ERROR: {e}")
        import traceback
        traceback.print_exc()
        error_count += 1

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Total threads checked: {len(result)}")
print(f"Successfully refreshed: {refreshed_count}")
print(f"Errors: {error_count}")
print()


## Reddit Crawler (Spidr)

Run the slow Reddit crawler to monitor subreddits for new posts.


In [ ]:
import sys
import time
import random
from pathlib import Path
from datetime import datetime

# Add src to path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.platforms.reddit import RedditAdapter
from feed.polling.engine import PollingEngine
from feed.storage.neo4j_connection import get_connection

# Configuration
MAX_CYCLES = 1  # Set to None for infinite loop, or a number for limited cycles
SUBREDDITS = [
    "AddisonRae",
    "BotezLive",
    "HannahBeast",
    "OfflinetvGirls",
    "Taylorhillfantasy",
    "SommerRay",
    "kendalljenner",
    "popheadscirclejerk",
    "WhatAWeeb",
    "ArianaGrande",
    "KatrinaBowden",
    "KiraKosarin",
    "KiraKosarinLewd",
    "LeightonMeester",
    "MargotRobbie",
    "MarinKitagawaR34",
    "McKaylaMaroney",
    "MelissaBenoist",
    "MinkaKelly",
    "MirandaKerr",
    "Models",
    "NatalieDormer",
    "NinaDobrev",
    "Nina_Agdal",
    "OfflinetvGirls",
    "OliviaRodrigoNSFW",
    "OneTrueMentalOut",
    "OvileeWorship",
    "PhoebeTonkin",
    "Pokimane",
    "PortiaDoubleday",
    "RachelCook",
    "RachelMcAdams",
    "SammiHanratty",
    "SaraSampaio",
    "SarahHyland",
    "SelenaGomez",
    "ShaileneWoodley",
    "StellaMaxwell",
    "SydneySweeney",
    "TOS_girls",
    "TaylorSwift",
    "TaylorSwiftCandids",
    "TaylorSwiftMidriff",
    "Taylorhillfantasy",
    "VanessaHudgens",
    "VolleyballGirls",
    "WatchItForThePlot",
    "angourierice",
    "annakendrick",
    "blakelively",
    "candiceswanepoel",
    "erinmoriartyNEW",
    "haydenpanettiere",
    "howdyhowdyyallhot",
    "islafisher",
    "jenniferlovehewitt",
    "jessicaalba",
    "karliekloss",
    "kateupton",
    "kayascodelario",
    "kristenbell",
    "kristinefroseth",
    "lizgillies",
    "milanavayntrub",
    "natalieportman",
    "oliviadunne",
    "sophieturner",
    "sunisalee",
    "victoriajustice",
    "victorious",
    "vsangels",
]

# Remove duplicates while preserving order
SUBREDDITS = list(dict.fromkeys(SUBREDDITS))

# Delay settings (in seconds)
REQUEST_DELAY_MIN = 10.0
REQUEST_DELAY_MAX = 30.0
SUBREDDIT_DELAY_MIN = 60.0
SUBREDDIT_DELAY_MAX = 180.0
CYCLE_DELAY_MIN = 300.0
CYCLE_DELAY_MAX = 600.0
STEP_DELAY_MIN = 5.0
STEP_DELAY_MAX = 15.0


def random_delay(min_sec: float, max_sec: float, reason: str = ""):
    """Sleep for a random duration and log it."""
    delay = random.uniform(min_sec, max_sec)
    if reason:
        print(f"  [DELAY] {reason}: {delay:.1f} seconds")
    time.sleep(delay)


print("=" * 80)
print("SLOW REDDIT CRAWLER - 100 Subreddits")
print("=" * 80)
print(f"Total subreddits: {len(SUBREDDITS)}")
print(f"Max cycles: {MAX_CYCLES if MAX_CYCLES else 'Infinite'}")
print(f"Request delays: {REQUEST_DELAY_MIN}-{REQUEST_DELAY_MAX} seconds")
print(f"Subreddit delays: {SUBREDDIT_DELAY_MIN}-{SUBREDDIT_DELAY_MAX} seconds")
print(f"Cycle delays: {CYCLE_DELAY_MIN}-{CYCLE_DELAY_MAX} seconds")
print("=" * 80)
print()

# Initialize Reddit adapter with slow delays
reddit = RedditAdapter(
    mock=False,
    delay_min=REQUEST_DELAY_MIN,
    delay_max=REQUEST_DELAY_MAX,
)

# Initialize Neo4j connection
try:
    neo4j = get_connection()
    print(f"Connected to Neo4j: {neo4j.uri}")
except Exception as e:
    print(f"Error connecting to Neo4j: {e}")
    print("Make sure NEO4J_URI and NEO4J_PASSWORD are set in .env")
    raise

# Initialize polling engine
engine = PollingEngine(neo4j, platform=reddit, dry_run=False)

cycle = 0

try:
    while True:
        if MAX_CYCLES and cycle >= MAX_CYCLES:
            print(f"\nReached max cycles ({MAX_CYCLES}). Stopping.")
            break
            
        cycle += 1
        print()
        print("=" * 80)
        print(f"CYCLE {cycle} - Starting at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        print()
        
        random_delay(STEP_DELAY_MIN, STEP_DELAY_MAX, "Pre-cycle delay")
        
        for idx, subreddit in enumerate(SUBREDDITS, 1):
            print()
            print("-" * 80)
            print(f"[{idx}/{len(SUBREDDITS)}] Processing r/{subreddit}")
            print("-" * 80)
            
            random_delay(STEP_DELAY_MIN, STEP_DELAY_MAX, "Pre-subreddit delay")
            
            try:
                # Poll the subreddit (only fetch first page to be slow)
                posts = engine.poll_source(
                    source=subreddit,
                    sort="new",
                    max_pages=1,
                    limit_per_page=100,
                )
                
                print(f"  Collected {len(posts)} posts from r/{subreddit}")
                
            except Exception as e:
                print(f"  ERROR processing r/{subreddit}: {e}")
                # Continue to next subreddit even on error
            
            # Delay between subreddits
            if idx < len(SUBREDDITS):
                random_delay(SUBREDDIT_DELAY_MIN, SUBREDDIT_DELAY_MAX, f"Between subreddits (next: r/{SUBREDDITS[idx]})")
        
        # Delay between cycles (if not at max)
        print()
        print("=" * 80)
        print(f"Cycle {cycle} complete. All {len(SUBREDDITS)} subreddits processed.")
        print(f"Completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        
        if not MAX_CYCLES or cycle < MAX_CYCLES:
            random_delay(CYCLE_DELAY_MIN, CYCLE_DELAY_MAX, "Between cycles")

except KeyboardInterrupt:
    print()
    print("=" * 80)
    print("Crawler stopped by user")
    print(f"Completed {cycle} cycle(s)")
    print("=" * 80)
except Exception as e:
    print()
    print("=" * 80)
    print(f"FATAL ERROR: {e}")
    print("=" * 80)
    import traceback
    traceback.print_exc()
    raise

print()
print("=" * 80)
print("Crawler finished")
print("=" * 80)


In [ ]:
# Query all subreddits linked to Taylor Swift
query = """
MATCH (c:Creator {slug: 'taylor-swift'})-[:HAS_SOURCE]->(s:Subreddit)
OPTIONAL MATCH (p:Post)-[:POSTED_IN]->(s)
OPTIONAL MATCH (p)-[:ABOUT]->(c)
RETURN s.name as subreddit, 
       count(DISTINCT p) as post_count,
       collect(DISTINCT p.era)[0..5] as eras
ORDER BY post_count DESC
"""

result = neo4j.execute_read(query)
print("Subreddits tracking Taylor Swift:")
print("=" * 80)
for record in result:
    eras = [e for e in record.get('eras', []) if e]
    era_str = f" (Eras: {', '.join(eras)})" if eras else ""
    print(f"\nr/{record.get('subreddit', 'N/A')}: {record.get('post_count', 0)} posts{era_str}")
